## Purpose of This Notebook

This notebook serves as an exploratory tool for examining the log files produced during the horse behavioural experiments conducted in October and November 2023. It facilitates querying the data that was pushed into the database in `logfile-to-database.ipynb`

In [ ]:
# | echo: false

import sys
from pathlib import Path
from pprint import pprint

import duckdb
import pandas as pd
from IPython.display import Markdown, display
from loguru import logger

In [ ]:
# | echo: false

import itables.options as opt
from itables import init_notebook_mode

init_notebook_mode(all_interactive=True)  # Display dataframes in a more friendly paginated manner
opt.pageLength = 20


In [ ]:
# | echo: false

# Custom notebook functions


def export_data_to_csv(df, info_message=None, filepath=None, header=None):
    df.to_csv(filepath, header=header)
    logger.info(f"{len(df)} rows - {info_message}: {filepath}")

def set_custom_logger_format():
    # custom_format = "<level>{level: <8}</level> | <cyan>{name}</cyan>:<cyan>{function}</cyan>:<cyan>{line}</cyan> - <level>{message}</level>"
    custom_format_no_func_info = "<level>{level: <8}</level> | <level>{message}</level>"
    logger.remove()    # Remove the default handler before adding a new one
    logger.add(sys.stderr, format=custom_format_no_func_info)

set_custom_logger_format()

### Directory information: log file data and outputs

In [ ]:
# | echo: false

LOGFILES_DIR = "../data/results/zips/data_17Jan2020_email_hillydale_equine"

assert Path(LOGFILES_DIR).exists()
logger.info(f"Logfiles dir: {LOGFILES_DIR}")


In [ ]:
# | echo: false

OUTPUT_DIR = "../data/results/notebooks"

assert Path(OUTPUT_DIR).exists()
logger.info(f"Outputs dir: {OUTPUT_DIR}")

In [ ]:
# | echo: false

DATA_DIR = "../data"

assert Path(DATA_DIR).exists()

DATA_DB  = Path(DATA_DIR) / "experiments_October2023.ddb"  # DuckDB name
logger.info(f"Database file: {DATA_DB}")

#### Connect to database

In [ ]:
# | echo: false

try:
    if DATA_DB.exists():  
        con = duckdb.connect(database=str(DATA_DB))
    else:
        logger.error(f"Database file: {DATA_DB} not found.")
except Exception as e:
    print("Error with Database file: {DATA_DB} - lock file?")
    logger.error(f"{e}")

In [ ]:
# Cross-check queries

con.sql("SHOW TABLES;")

##### Reconstruct an experiment example

In [ ]:
experiments_df = con.sql("SELECT * FROM Experiments").df()

In [ ]:
experiments_df

In [ ]:
# | echo: false

# experiments_df.sample()

# ExperimentID = 128 
# LogFilename = "Experiment_2023-11-15T05:13:56.627916_gio_2.0_RPE-A.log"

In [ ]:
con.sql(
    """
    SELECT
        ExperimentID, 
        SubjectName, 
        SubjectNumber, 
        SessionNumber,
        ExperimentType, 
        Comment, 
        DateTime, 
        LogFileName
    FROM Experiments 
    WHERE ExperimentID = 128;
"""
).df()

In [ ]:
con.sql("""
    SELECT 
        -- Trials.TrialID,
        Trials.TrialNumber,
        -- TrialStartTime,
        -- TrialEndTime,
        -- EventID, 
        EventNumber, 
        EventTime, 
        EventType,
        EventElapsedTime
    FROM Experiments
    INNER JOIN Trials ON Experiments.ExperimentID = Trials.ExperimentID
    INNER JOIN Events ON Trials.TrialID = Events.TrialID
    WHERE Experiments.ExperimentID = 128;
""").df()

In [ ]:
# !open ../data/results/zips/data_17Jan2020_email_hillydale_equine/Experiment_2023-11-15T05:13:56.627916_gio_2.0_RPE-A.log

In [ ]:
con.sql("""
    SELECT 
        -- Experiments.SubjectName,
        -- Experiments.SessionNumber,
        -- Trials.TrialID,
        Trials.TrialNumber,
        -- TrialStartTime,
        -- TrialEndTime,
        -- EventID, 
        EventNumber, 
        EventTime, 
        EventType,
        EventElapsedTime
    FROM Experiments
    INNER JOIN Trials ON Experiments.ExperimentID = Trials.ExperimentID
    INNER JOIN Events ON Trials.TrialID = Events.TrialID
    WHERE Experiments.SubjectName = 'gio' AND  Experiments.SessionNumber = 2;
""").df()